pip install torch torchvision
pip install git+https://github.com/openai/CLIP.git

In [2]:
import torch
import clip
from PIL import Image
import requests # Для загрузки изображения по URL
from io import BytesIO # Для работы с изображением из URL

# Загрузка модели CLIP
# Доступные модели можно посмотреть тут: clip.available_models()
# 'ViT-B/32' - это одна из стандартных моделей
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

print(f"Модель загружена на: {device}")

# 1. Подготовка изображения
# Ты можешь использовать URL изображения из интернета или путь к локальному файлу
# Пример с URL:
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg" # Пример: изображение с кошкой и ноутбуком
try:
    response = requests.get(image_url)
    response.raise_for_status() # Проверка на ошибки HTTP
    image_pil = Image.open(BytesIO(response.content)).convert("RGB")
    print("Изображение успешно загружено по URL.")
except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке изображения по URL: {e}")
    # Если URL не работает, можно использовать локальный файл:
    # try:
    #     image_path = "path/to/your/image.jpg" # Замени на путь к своему изображению
    #     image_pil = Image.open(image_path).convert("RGB")
    #     print(f"Изображение успешно загружено из файла: {image_path}")
    # except FileNotFoundError:
    #     print(f"Файл изображения не найден: {image_path}")
    #     exit()
    # except Exception as e_file:
    #     print(f"Ошибка при открытии локального файла: {e_file}")
    #     exit()
    exit()


# Препроцессинг изображения для CLIP
image_input = preprocess(image_pil).unsqueeze(0).to(device)

# 2. Вопрос и варианты ответов
question = "What is on the laptop?"
candidate_answers = [
    "a cat",
    "a dog",
    "there is no laptop",
    "a remote control"
]
# (Для этого изображения правильный ответ "a cat")

# 3. Подготовка текстовых описаний для CLIP
# Мы будем оценивать, насколько каждый "ответ" соответствует изображению
text_descriptions = [f"A photo of {answer}" for answer in candidate_answers]
# Или можно просто использовать сами ответы, если они достаточно описательны:
# text_descriptions = candidate_answers

text_tokens = clip.tokenize(text_descriptions).to(device)

# 4. Получение признаков (эмбеддингов) от CLIP
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_tokens)

# Нормализация признаков (важно для косинусного сходства)
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)

# 5. Вычисление сходства между изображением и каждым вариантом ответа
# Косинусное сходство можно вычислить как матричное произведение нормализованных векторов
similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
# scores = similarity[0].tolist() # Получаем список вероятностей

# Находим наиболее вероятный ответ
best_answer_index = similarity.argmax().item()
predicted_answer = candidate_answers[best_answer_index]
confidence_score = similarity[0, best_answer_index].item()

print(f"\nВопрос: {question}")
print("Варианты ответов:")
for i, answer in enumerate(candidate_answers):
    print(f"  {i+1}. {answer} (Сходство: {similarity[0, i].item():.4f})")

print(f"\nПредсказанный ответ: {predicted_answer}")
print(f"Уверенность: {confidence_score:.4f}")

100%|███████████████████████████████████████| 338M/338M [02:24<00:00, 2.44MiB/s]


Модель загружена на: cpu
Изображение успешно загружено по URL.

Вопрос: What is on the laptop?
Варианты ответов:
  1. a cat (Сходство: 0.5319)
  2. a dog (Сходство: 0.0027)
  3. there is no laptop (Сходство: 0.1119)
  4. a remote control (Сходство: 0.3534)

Предсказанный ответ: a cat
Уверенность: 0.5319


In [4]:
import torch
import clip
from PIL import Image
import requests
from io import BytesIO

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print(f"Модель CLIP загружена на: {device}")

def get_clip_vqa_answer(image_path_or_url, question, candidate_answers):
    try:
        if image_path_or_url.startswith("http"):
            response = requests.get(image_path_or_url)
            response.raise_for_status()
            image_pil = Image.open(BytesIO(response.content)).convert("RGB")
        else:
            image_pil = Image.open(image_path_or_url).convert("RGB")
    except Exception as e:
        print(f"Ошибка загрузки/открытия изображения {image_path_or_url}: {e}")
        return None, None

    image_input = preprocess(image_pil).unsqueeze(0).to(device)
    
    text_descriptions = [f"A photo of {ans}" for ans in candidate_answers]

    text_tokens = clip.tokenize(text_descriptions).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_tokens)

    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
    
    best_answer_index = similarity.argmax().item()
    predicted_answer = candidate_answers[best_answer_index]
    confidence_score = similarity[0, best_answer_index].item()
    
    return predicted_answer, confidence_score, similarity[0].tolist(), image_pil

# --- Тестового набора данных ---
test_data = [
    {
        "image": "http://images.cocodataset.org/val2017/000000039769.jpg", # Два кота с пульмати на розовом пледе
        "question": "How many cats are in the picture?",
        "candidates": ["one cat", "two cats", "three cats", "no cats"],
        "correct_answer": "two cats"
    },
    {
        "image": "http://images.cocodataset.org/val2017/000000039769.jpg", # То же изображение
        "question": "What is the color of the blanket?",
        "candidates": ["pink", "blue", "green", "white"],
        "correct_answer": "pink"
    },
    {
        "image": "http://images.cocodataset.org/val2017/000000039769.jpg", # То же изображение
        "question": "What is on the laptop?",
        "candidates": ["a cat", "a dog", "a remote control", "a book"],
        "correct_answer": "a cat"
    },
]

# --- Проведение эксперимента ---
correct_predictions = 0
results_for_report = []

for item in test_data:
    print(f"\nОбработка изображения: {item['image']}")
    predicted_answer, confidence, all_scores, pil_image = get_clip_vqa_answer(item["image"], item["question"], item["candidates"])
    
    if predicted_answer: # Если изображение загрузилось и ответ получен
        is_correct = (predicted_answer == item["correct_answer"])
        if is_correct:
            correct_predictions += 1
        
        results_for_report.append({
            "image_src": item["image"],
            "question": item["question"],
            "candidates_scores": list(zip(item["candidates"], [f"{s:.4f}" for s in all_scores])),
            "predicted_answer": predicted_answer,
            "correct_answer": item["correct_answer"],
            "is_correct": is_correct,
            "confidence": f"{confidence:.4f}",

        })
        print(f"Вопрос: {item['question']}")
        print(f"Кандидаты и их сходство: {results_for_report[-1]['candidates_scores']}")
        print(f"Предсказано: {predicted_answer} (Уверенность: {confidence:.4f}) | Правильно: {item['correct_answer']} | Результат: {'Верно' if is_correct else 'Неверно'}")

accuracy = (correct_predictions / len(test_data)) * 100 if test_data else 0
print(f"\nОбщая точность на тестовом наборе: {accuracy:.2f}% ({correct_predictions}/{len(test_data)})")


Модель CLIP загружена на: cpu

Обработка изображения: http://images.cocodataset.org/val2017/000000039769.jpg
Вопрос: How many cats are in the picture?
Кандидаты и их сходство: [('one cat', '0.0207'), ('two cats', '0.7262'), ('three cats', '0.2333'), ('no cats', '0.0198')]
Предсказано: two cats (Уверенность: 0.7262) | Правильно: two cats | Результат: Верно

Обработка изображения: http://images.cocodataset.org/val2017/000000039769.jpg
Вопрос: What is the color of the blanket?
Кандидаты и их сходство: [('pink', '0.5093'), ('blue', '0.2395'), ('green', '0.0489'), ('white', '0.2023')]
Предсказано: pink (Уверенность: 0.5093) | Правильно: pink | Результат: Верно

Обработка изображения: http://images.cocodataset.org/val2017/000000039769.jpg
Вопрос: What is on the laptop?
Кандидаты и их сходство: [('a cat', '0.5980'), ('a dog', '0.0031'), ('a remote control', '0.3973'), ('a book', '0.0016')]
Предсказано: a cat (Уверенность: 0.5980) | Правильно: a cat | Результат: Верно

Общая точность на тестов